# 🕵️ Rewriting Biographies

Instead of replacing entities with tokens, rewrite mode generates a
privacy-safe transformation of the entire text. The pipeline:

1. Detects entities (same as replace mode, plus latent entity detection)
2. Classifies the domain and assigns sensitivity dispositions
3. Generates a rewritten version that obscures sensitive entities
4. Evaluates quality (utility) and privacy (leakage) with an automated repair loop

After `run()`, call `Anonymizer.evaluate()` for optional LLM-as-judge scoring.

#### 📚 What you'll learn

- Configure rewrite mode with `PrivacyGoal` to specify what to protect and what to preserve
- Set evaluation criteria and risk tolerance for automated quality checks
- Preview rewritten text and inspect utility / leakage scores
- Triage flagged records with `needs_human_review`
- Run `evaluate()` for detection validity and holistic judge scores (privacy, quality, style)

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
  - The default `build.nvidia.com` (NVIDIA Build) setup is a convenient way to try Anonymizer and iterate on previews. Use of NVIDIA Build is subject to NVIDIA Build's own terms of service and privacy practices, which are separate from and independent of the NeMo Framework library. NVIDIA Build is intended for evaluation and testing purposes only and may not be used in production environments. Do not upload any confidential information or personal data when using NVIDIA Build. Your use of NVIDIA Build is logged for security purposes and to improve NVIDIA products and services.
  - Request and token rate limits on `build.nvidia.com` vary by account and model access, and lower-volume development access can be slow for full-dataset runs. Start with `preview()` on a small sample, then move to your own endpoint for production data and usage.
- Import `Rewrite` and `PrivacyGoal`.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.
- `configure_logging(LoggingConfig.default())` keeps logs at INFO. Switch to `LoggingConfig.debug()` when troubleshooting.


In [1]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [ ]:
from anonymizer import (
    Anonymizer,
    AnonymizerConfig,
    AnonymizerInput,
    LoggingConfig,
    PrivacyGoal,
    Rewrite,
    configure_logging,
)

configure_logging(LoggingConfig.default())

In [3]:
anonymizer = Anonymizer()

[16:06:37] [INFO] 🔧 Anonymizer initialized with 3 model configs
[16:06:37] [INFO]   |-- 🔎 detector:  gliner-pii-detector
[16:06:37] [INFO]   |-- ✅ validator: gpt-oss-120b
[16:06:37] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Input data

- Same biographies dataset used in earlier notebooks -- familiar data makes it
  easy to compare rewrite output against replace output.

In [4]:
input_data = AnonymizerInput(
    source="https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)

## 🎛️ Configure

- `PrivacyGoal` spells out what to **protect** and what to **preserve** --
  this gives the rewriter clear, domain-specific guidance.
- `risk_tolerance` (default `"low"`) and `max_repair_iterations` (default `3`)
  control the automated quality gate --
  see [Risk tolerance](../../concepts/rewrite/#risk-tolerance) for presets.

In [5]:
config = AnonymizerConfig(
    rewrite=Rewrite(
        privacy_goal=PrivacyGoal(
            protect="All direct identifiers and quasi-identifier combinations (names, locations, employers, dates)",
            preserve="Career trajectory, educational background, and professional accomplishments",
        ),
    ),
)

## 👁️ Preview

- `preview()` runs on a small sample so you can iterate on privacy goals
  and evaluation criteria before committing to a full run.

In [6]:
preview = anonymizer.preview(
    config=config,
    data=input_data,
    num_records=3,
)

preview.display_record(0)

[16:06:46] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')
[16:06:46] [INFO] 🔍 Running entity detection on 3 records
[16:07:58] [INFO]   |-- 📋 Detection complete — 78 entities found across 3 records (0 failed) [72.5s]
[16:07:58] [INFO]   |-- labels: first_name=22, state=6, organization_name=6, age=5, occupation=5, city=5, political_view=4, last_name=3, race_ethnicity=3, language=3, company_name=3, degree=2, field_of_study=2, education_level=2, street_address=2, place_name=1, date_of_birth=1, project_name=1, employment_status=1, religious_belief=1
[16:07:58] [INFO] ✏️ Running rewrite pipeline
[16:10:14] [INFO] Evaluate-repair loop: all rows pass at iteration 0
[16:10:32] [INFO]   |-- 📋 Rewrite complete (0 failed) [154.1s]
[16:10:32] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Entity,Label,Sensitivity,Protection
Bobby,first_name,high,replace
Watford,last_name,high,replace
Maya,first_name,high,replace
Aria and Leo,first_name,high,replace
Mexican,race_ethnicity,high,generalize
Denver,city,medium,generalize
University of Colorado Boulder,organization_name,medium,generalize
Colorado Veterinary Clinic,organization_name,medium,generalize
VCA Animal Hospital,company_name,medium,generalize
Christian Democrat,political_view,medium,generalize


In [7]:
preview.display_record(1)

Entity,Label,Sensitivity,Protection
204 Bluegrass,street_address,high,replace
Jodi,first_name,high,replace
Allison,last_name,high,replace
36,age,high,generalize
4,age,high,remove
7,age,high,remove
Alex,first_name,high,replace
BA,education_level,low,leave_as_is
Caucasian,race_ethnicity,low,leave_as_is
Clayton,city,high,generalize


## 🚀 Full run

- `result.dataframe` has user-facing columns: rewritten text, scores, and the review flag.
- `result.trace_dataframe` has every intermediate column for debugging.

In [8]:
result = anonymizer.run(config=config, data=input_data)

result.dataframe.head()

[14:51:45] [INFO] 📂 Loaded 25 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')
[14:51:45] [INFO] 🔍 Running entity detection on 25 records


[14:54:03] [INFO]   |-- 📋 Detection complete — 645 entities found across 25 records (0 failed) [137.9s]
[14:54:03] [INFO]   |-- labels: first_name=152, occupation=47, city=44, company_name=43, organization_name=31, race_ethnicity=30, state=30, education_level=30, last_name=26, age=26, religious_belief=26, political_view=25, street_address=23, university=22, language=21, field_of_study=14, place_name=10, county=10, date_of_birth=9, employment_status=9, degree=7, date=5, project_name=1, full_name=1, country=1, gender=1, postcode=1
[14:54:03] [INFO] ✏️ Running rewrite pipeline
[15:12:06] [INFO] Evaluate-repair loop iteration 0: 7/25 rows need repair
[15:12:53] [INFO] Evaluate-repair loop: all rows pass at iteration 1
[15:13:39] [INFO]   |-- 📋 Rewrite complete (0 failed) [1076.7s]
[15:13:39] [INFO] 🎉 Pipeline complete — 25 records processed, 0 total failures


,biography,biography_rewritten,utility_score,leakage_mass,weighted_leakage_rate,any_high_leaked,needs_human_review
0,"Bobby Watford, a 40‑year‑old Mexican veterinar...","Ethan Hawkins, a 40‑year‑old Mexican veterinar...",1.0,0.0,0.0,False,False
1,Idilio Bell is a 37‑year‑old astronomer living...,Rafael Kline is a 37‑year‑old astronomer livin...,0.808333,0.0,0.0,False,False
2,"Jodi Allison, 36, lives at 204 Bluegrass in Cl...","Tara Kendall, 36, lives at a street address in...",0.877778,0.0,0.0,False,False
3,James Mills is a 69‑year‑old paramedic who liv...,Victor Hawthorne is a 69‑year‑old paramedic wh...,0.909091,0.0,0.0,False,False
4,Nancy Burton is a 21‑year‑old cashier who live...,Maya Hawthorne is a 21‑year‑old cashier who li...,0.957692,0.0,0.0,False,False


In [9]:
result.dataframe[["biography_rewritten", "utility_score", "leakage_mass", "needs_human_review"]].head()

,biography_rewritten,utility_score,leakage_mass,needs_human_review
0,"Ethan Hawkins, a 40‑year‑old Mexican veterinar...",1.0,0.0,False
1,Rafael Kline is a 37‑year‑old astronomer livin...,0.808333,0.0,False
2,"Tara Kendall, 36, lives at a street address in...",0.877778,0.0,False
3,Victor Hawthorne is a 69‑year‑old paramedic wh...,0.909091,0.0,False
4,Maya Hawthorne is a 21‑year‑old cashier who li...,0.957692,0.0,False


In [10]:
result.trace_dataframe.columns.tolist()

['biography',
 '_anonymizer_record_id',
 '_raw_detected_entities',
 '_seed_entities',
 '_tag_notation',
 '_seed_validation_candidates',
 '_seed_tagged_text',
 '_validated_entities',
 '_seed_entities_json',
 '_initial_tagged_text',
 '_validated_seed_entities',
 '_augmented_entities',
 '_merged_entities',
 '_merged_tagged_text',
 '_validation_candidates',
 '_detected_entities',
 'biography_with_spans',
 '_latent_entities',
 'final_entities',
 '_entities_by_value',
 '_replacement_map',
 '_domain',
 '_domain_supplement',
 '_domain_supplement_privacy',
 '_sensitivity_disposition',
 '_privacy_qa',
 '_sensitivity_disposition_block',
 '_rewrite_disposition_block',
 '_replacement_map_for_prompt',
 '_full_rewrite',
 'biography_rewritten',
 '_meaning_units',
 '_meaning_units_serialized',
 '_quality_qa',
 '_repair_iterations',
 '_quality_qa_reanswer',
 '_quality_qa_compare',
 '_privacy_qa_reanswer',
 'utility_score',
 'leakage_mass',
 'weighted_leakage_rate',
 'any_high_leaked',
 '_needs_repair',


## 🚩 Filter by review flag

- Records where automated metrics exceed thresholds are flagged for manual review.
- Use this to prioritize human attention on the records that need it most.
- See [Working with flagged records](../../concepts/rewrite/#working-with-flagged-records)
  for guidance on diagnosing and resolving flagged records.

In [11]:
df = result.dataframe
flagged = df[df["needs_human_review"] == True]  # noqa: E712
print(f"{len(flagged)} of {len(df)} records flagged for human review")
flagged.head()

0 of 25 records flagged for human review


,biography,biography_rewritten,utility_score,leakage_mass,weighted_leakage_rate,any_high_leaked,needs_human_review


## 🔬 Evaluate (optional)

Call `evaluate()` to run LLM-as-judge scoring on the rewrite result — detection validity and three quality rubrics (privacy, quality, style).
See [Evaluation](../../concepts/evaluation/#rewrite-evaluation) for details.

In [ ]:
evaluated = anonymizer.evaluate(result)

In [ ]:
evaluated.display_record(0)

## ⏭️ Next steps

- **[⚖️ Rewriting Legal Documents](../05_rewriting_legal_documents/)** --
  rewrite legal text with custom entity labels and domain-specific privacy goals.
- **[📊 Evaluation](../../concepts/evaluation/#rewrite-evaluation)** --
  learn about the detection validity and rewrite quality judges in detail.
- **[🎯 Choosing a Replacement Strategy](../03_choosing_a_replacement_strategy/)** --
  compare Redact, Annotate, Hash, and Substitute if you prefer token-level replacement.
- **[🔍 Inspecting Detected Entities](../02_inspecting_detected_entities/)** --
  debug what the detection pipeline found before rewriting.